# SOLiD-PINN v0.8.0 — Targeted experimental evidence export
Use the existing Drive cache from Inventory V2. No network downloads. Verify the four cached file SHA-256 digests before exporting selected worksheets without changing cell values, rows, columns, or units. These are public publisher data, not the private Dessantis inputs. No fit or training. Output is a compact ZIP for scientific audit; open-data attribution remains in manifest.

In [ ]:
from google.colab import drive
from pathlib import Path
import csv, hashlib, json, zipfile, shutil, sys, subprocess, tempfile
drive.mount('/content/drive')
subprocess.run([sys.executable,"-m","pip","install","openpyxl==3.1.5"],check=True)
from openpyxl import load_workbook
BASE=Path('/content/drive/MyDrive/SOLiD-PINN/open_benchmarks/downloads')
OUTPUT=Path('/content/SOLiD-PINN_open_evidence.zip')
SPECS=[
("umd/master.csv","f589036df04800ee164f657be05cfb47e1f60d8a289c12f70c1880f7944b1688","https://drum.lib.umd.edu/items/156429b8-2dc2-4078-b293-8d33e077bc91","CC0-1.0",{}),
("puls2024/cycling.xlsx","9ea011e59859fad7f5821ccc2e0bc39d69ae9e57da192ff3f1d00638b0727bce","https://doi.org/10.1038/s41560-024-01634-3","Article CC BY 4.0; confirm file-specific notices",{"Processing & Summary":"processing_summary.csv","Supp. Fig. 2":"discharge_capacity.csv","Spec. charge capacity":"charge_capacity.csv"}),
("liopa/main.xlsx","c61a67cbf13540858a58a341a1d27ae4c0fce23b7e4f810a41b8168271218420","https://doi.org/10.5281/zenodo.17816356","CC-BY-4.0",{"SP F2":"sp_main.csv","FC F3":"full_cell_main.csv","Cycling FC F3":"cycling_main.csv","Dry room Ion cond F2":"conductivity_relative.csv"}),
("liopa/supporting.xlsx","22640fa76a00d90626cfe8fc2f3d4fc20ee71c3f87aba865f064960f1e3cd05e","https://doi.org/10.5281/zenodo.17816356","CC-BY-4.0",{"Li SP & ASR S9 S10":"sp_asr.csv","EIS S5":"eis.csv","EIS humidity test S5":"eis_humidity.csv","FC S12":"full_cell_supporting.csv"})
]
def sha(path):
    h=hashlib.sha256()
    with path.open("rb") as f:
        for b in iter(lambda:f.read(1024*1024),b""):h.update(b)
    return h.hexdigest()
for rel,expected,*_ in SPECS:
    p=BASE/rel
    if not p.is_file() or sha(p)!=expected: raise RuntimeError(f"Missing or changed source: {rel}. Restore the original cached file.")
print("4/4 original cached source hashes match.")

In [ ]:
manifest={"schema":1,"purpose":"targeted source evidence; no normalization, fitting or model-ready claim","sources":[]}
with tempfile.TemporaryDirectory() as temp:
    root=Path(temp)
    for rel,expected,url,license,sheets in SPECS:
        p=BASE/rel
        folder=root/Path(rel).parent
        folder.mkdir(exist_ok=True,parents=True)
        entry={"source_file":rel,"sha256":expected,"source_url":url,"license":license,"exports":[]}
        if not sheets:
            dest=folder/"master.csv"
            shutil.copyfile(p,dest)
            entry["exports"].append({"path":str(dest.relative_to(root)),"sha256":sha(dest),"note":"byte-exact copy; CP1252 provisional"})
        else:
            book=load_workbook(p,read_only=True,data_only=True)
            try:
                for sheet,name in sheets.items():
                    ws=book[sheet]
                    dest=folder/name
                    count=0
                    with dest.open("w",encoding="utf-8",newline="") as f:
                        w=csv.writer(f)
                        for row in ws.iter_rows(values_only=True):
                            w.writerow(row)
                            count+=1
                    entry["exports"].append({"sheet":sheet,"path":str(dest.relative_to(root)),"sha256":sha(dest),"rows_written":count,"declared_columns":ws.max_column,"note":"original sheet layout, including blanks and headers; cached formula values; no unit conversions"})
                    print(rel,sheet,count)
            finally:
                book.close()
        manifest["sources"].append(entry)
    (root/"manifest.json").write_text(json.dumps(manifest,indent=2)+"\n")
    tmp=OUTPUT.with_suffix(".part")
    with zipfile.ZipFile(tmp,"w",zipfile.ZIP_DEFLATED) as z:
        for path in sorted(root.rglob("*")):
            if path.is_file():z.write(path,path.relative_to(root))
    tmp.replace(OUTPUT)
print("Export complete:",OUTPUT,OUTPUT.stat().st_size,"bytes")

Upload the resulting ZIP in ChatGPT. No need to download the failed Nature workbooks or repeat the inventory. Do not commit raw workbooks or this ZIP to GitHub without reviewing file-specific redistribution conditions.

In [ ]:
from google.colab import files
files.download(str(OUTPUT))